# Amma Sewana — Gemma 4 E2B fine-tuning (clean build)

LoRA fine-tune of **Gemma 4 E2B** on 301 instruction-output pairs (English + Sinhala + Tamil) drawn from Sri Lanka MOH 2013, NICE NG133/NG3/NG201, and RCOG GTG 52/47/57.

**Hackathon:** Gemma 4 Good (Google DeepMind / Kaggle). Fine-tuning Gemma 4 — not Gemma 3.

This notebook has every environment fix already applied (Gemma 4 multimodal processor, SDPA attention, pre-tokenisation, plain `Trainer`, no on-disk checkpoints). Run top to bottom, once.

## Before running
1. **Settings → Accelerator: GPU T4 ×2**
2. **Settings → Internet: ON** (needs phone-verified Kaggle account)
3. **Add Input →** attach the dataset `sri-lanka-maternal-care-qa-dataset`
4. **Add-ons → Secrets →** add `HF_TOKEN` (a HuggingFace **write** token) and tick Attached

Runtime: ~30-40 min for 3 epochs.

## 1. Install — pinned, clean order

Uninstall first so Kaggle's pre-baked `huggingface_hub` / `transformers` can't shadow the versions Unsloth needs (avoids the `KernelInfo` ImportError).

In [ ]:
%%capture
!pip uninstall -y huggingface_hub unsloth unsloth_zoo transformers
!pip install --no-cache-dir --upgrade "huggingface_hub" "transformers"
!pip install --no-cache-dir --upgrade "unsloth[kaggle-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps --upgrade "trl<0.9.0" peft accelerate bitsandbytes datasets

In [ ]:
import torch
print("CUDA:", torch.cuda.is_available(), "| GPUs:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(" ", torch.cuda.get_device_properties(i).name)

## 2. Load Gemma 4 E2B

`attn_implementation="sdpa"` is required — Gemma 4's default FlexAttention path crashes under torch.compile inside Unsloth.

If you still hit a `torch._dynamo` / FlexAttention error, add a cell *above the install cell* with:
`import os; os.environ["UNSLOTH_COMPILE_DISABLE"]="1"; os.environ["TORCHDYNAMO_DISABLE"]="1"`

In [ ]:
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LEN = 2048

MODEL_CANDIDATES = [
    "unsloth/gemma-4-E2B-it",
    "unsloth/gemma-4-2b-it",
    "google/gemma-4-2b-it",
    "google/gemma-4-e2b-it",
]

model, tokenizer, last_err = None, None, None
for name in MODEL_CANDIDATES:
    try:
        print(f"Trying {name} ...")
        model, tokenizer = FastLanguageModel.from_pretrained(
            model_name=name,
            max_seq_length=MAX_SEQ_LEN,
            dtype=None,
            load_in_4bit=True,
            attn_implementation="sdpa",
        )
        print("Loaded:", name)
        break
    except Exception as e:
        last_err = e
        print("  failed:", e)

if model is None:
    raise RuntimeError("No Gemma 4 model id loaded — check huggingface.co/unsloth for the current id.") from last_err

## 3. Attach LoRA adapters

Run this cell exactly once. Re-running it errors with "already added LoRA adapters" — if that happens, Factory reset and start over.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

from peft import PeftModel
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("Is PeftModel:", isinstance(model, PeftModel), "| trainable params:", f"{trainable:,}")
assert isinstance(model, PeftModel), "LoRA wrap failed"

## 4. Load the dataset

In [ ]:
from pathlib import Path
from datasets import load_dataset

CANDIDATE_PATHS = [
    "/kaggle/input/sri-lanka-maternal-care-qa-dataset/dataset.jsonl",
    "/kaggle/input/amma-sewana-dataset/dataset.jsonl",
    "./dataset.jsonl",
]
dataset_path = next((p for p in CANDIDATE_PATHS if Path(p).exists()), None)
if dataset_path is None:
    raise FileNotFoundError(f"dataset.jsonl not found in {CANDIDATE_PATHS} — attach the Kaggle dataset.")
print("Using:", dataset_path)

raw = load_dataset("json", data_files=dataset_path, split="train")
print(raw)
print("Languages:", raw.unique("lang"))

## 5. Format with Gemma chat template + pre-tokenise

Gemma 4 is multimodal — the object in `tokenizer` is a `Gemma4Processor`. We pull out its inner text tokenizer (`text_tokenizer`) and tokenise the data ourselves, so the broken processor `__call__` path is never used during training.

In [ ]:
SYSTEM_INSTRUCTION = (
    "You are Amma Sewana, an AI assistant for Sri Lanka Public Health Midwives (PHMs). "
    "Help PHMs assess pregnant mothers and answer clinical questions. "
    "Follow Sri Lanka Ministry of Health maternal care guidelines, NICE and RCOG guidelines. "
    "Always state risk level (LOW/MEDIUM/HIGH/URGENT) and immediate action where relevant. "
    "Reply in the same language as the question (Sinhala, Tamil, or English)."
)

text_tokenizer = tokenizer.tokenizer if hasattr(tokenizer, "tokenizer") else tokenizer
text_tokenizer.padding_side = "right"
if text_tokenizer.pad_token is None:
    text_tokenizer.pad_token = text_tokenizer.eos_token

def to_text(example):
    user = f"{SYSTEM_INSTRUCTION}\n\n{example['instruction']}"
    msgs = [
        {"role": "user", "content": user},
        {"role": "assistant", "content": example["output"]},
    ]
    return {"text": text_tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)}

def tokenize_fn(batch):
    return text_tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_SEQ_LEN,
        padding=False,
        add_special_tokens=False,
    )

ds_text = raw.map(to_text, remove_columns=raw.column_names)
ds_tok = ds_text.map(tokenize_fn, batched=True, remove_columns=["text"])
print("samples:", len(ds_tok), "| first length:", len(ds_tok[0]["input_ids"]))
print(ds_text[0]["text"][:400])

## 6. Train

Plain HuggingFace `Trainer` (SFTTrainer mis-handles the Gemma 4 processor). `save_strategy="no"` — nothing is written to disk during training, so Kaggle's disk never fills. The model lives in memory and is pushed straight to HuggingFace afterwards.

In [ ]:
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling

args = TrainingArguments(
    output_dir="/kaggle/working/amma_out",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    warmup_steps=20,
    num_train_epochs=3,
    learning_rate=2e-4,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=5,
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    seed=42,
    report_to="none",
    save_strategy="no",
    remove_unused_columns=False,
)

collator = DataCollatorForLanguageModeling(tokenizer=text_tokenizer, mlm=False)

trainer = Trainer(
    model=model,
    processing_class=text_tokenizer,
    train_dataset=ds_tok,
    data_collator=collator,
    args=args,
)

trainer_stats = trainer.train()
print("Final loss:", trainer_stats.training_loss)

## 7. Smoke test

Confirms the adapter trained: answers should cite sources (e.g. "Source: RCOG GTG 57") and give risk levels.

In [ ]:
FastLanguageModel.for_inference(model)

def chat(question, max_new_tokens=400):
    user = f"{SYSTEM_INSTRUCTION}\n\n{question}"
    prompt = text_tokenizer.apply_chat_template(
        [{"role": "user", "content": user}],
        tokenize=False, add_generation_prompt=True,
    )
    inputs = text_tokenizer(prompt, return_tensors="pt").to(model.device)
    out = model.generate(**inputs, max_new_tokens=max_new_tokens,
                         temperature=0.3, top_p=0.95, top_k=40,
                         do_sample=True, repetition_penalty=1.05)
    return text_tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

for q in [
    "ගර්භණී වෙලා මාස 7යි. දරුවාගේ සෙලවීම අඩුයි කියලා හිතෙනවා. මොකද කරන්නේ?",
    "எனது இரத்த அழுத்தம் 162/108. வயிற்றின் மேல் வலதுபுறம் வலி உள்ளது. என் நிலை என்ன?",
    "32-year-old G3P2, 34 weeks, BP 162/108, proteinuria 2+, severe headache. Immediate action?",
]:
    print("Q:", q)
    print("A:", chat(q))
    print("---\n")

## 8. Push the LoRA adapter to HuggingFace

Adapter only (~50 MB). Do **not** push a merged model — the 10 GB merged checkpoint overflows Kaggle's disk. Base model + this adapter is fully reproducible.

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

token = UserSecretsClient().get_secret("HF_TOKEN")
login(token=token)

HF_REPO = "Manjula808/amma-sewana-gemma4-lora"  # <-- your HF username (case-sensitive)
model.push_to_hub(HF_REPO, text_tokenizer, token=token)
print("Adapter pushed:", f"https://huggingface.co/{HF_REPO}")

## 9. Save the adapter to notebook output (optional backup)

Small enough (~50 MB) to keep as a Kaggle output artifact too.

In [ ]:
model.save_pretrained("/kaggle/working/amma_sewana_lora_adapter")
text_tokenizer.save_pretrained("/kaggle/working/amma_sewana_lora_adapter")
print("Saved to /kaggle/working/amma_sewana_lora_adapter")
!du -sh /kaggle/working/amma_sewana_lora_adapter

## Done

When this notebook runs clean end-to-end: **Save Version → Save & Run All (Commit)**. That committed version is the reproducible artifact for the Unsloth prize.

Next: build the before/after evaluation (base Gemma 4 vs this fine-tune) to produce the accuracy table for the writeup.